In [3]:
import pandas as pd
import os
from IPython.display import display, clear_output
import ipywidgets as widgets
from PIL import Image
from pathlib import Path

# ==========================================
# 1. 경로 설정
# ==========================================
IMG_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/4. core추출 v1.3_1088"
EXCEL_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/3. ROI_LETTUCE/1.5작기_lettuce/4. core추출 v1.3_1088/core_v1_3_1088_sorted.xlsx"

# ==========================================
# 2. 데이터 로드 및 전처리
# ==========================================
df = pd.read_excel(EXCEL_PATH)
# 실제 파일 경로 매칭을 위해 이미지 폴더의 파일 리스트 확보
all_files = os.listdir(IMG_DIR)
file_map = {Path(f).stem: f for f in all_files}

# 전역 변수 (상태 관리용)
filtered_df = pd.DataFrame()
current_idx = 0

# ==========================================
# 3. 위젯 생성
# ==========================================
strength_input = widgets.IntText(value=2, description='Core Level:', style={'description_width': 'initial'})
search_input = widgets.Text(placeholder='파일명.png 입력 후 엔터', description='검색:', style={'description_width': 'initial'})
prev_btn = widgets.Button(description='< 이전', button_style='info')
next_btn = widgets.Button(description='다음 >', button_style='info')
out = widgets.Output()

# ==========================================
# 4. 기능 함수
# ==========================================
def display_image():
    global current_idx
    with out:
        clear_output(wait=True)
        if filtered_df.empty:
            print(f"해당 core_strength({strength_input.value})를 가진 데이터가 없습니다.")
            return

        row = filtered_df.iloc[current_idx]
        img_id = row['image_id']
        real_file_name = file_map.get(img_id, None)

        print(f"[{current_idx + 1} / {len(filtered_df)}] 파일명: {real_file_name}")
        print(f"Excel ID: {img_id} | Core Strength: {row['core_strength_v2']}")

        if real_file_name:
            img_path = os.path.join(IMG_DIR, real_file_name)
            img = Image.open(img_path)
            # 화면 크기에 맞게 리사이즈 (표시용)
            img.thumbnail((600, 600))
            display(img)
        else:
            print("❌ 폴더 내에서 파일을 찾을 수 없습니다.")

def on_filter_change(change):
    global filtered_df, current_idx
    filtered_df = df[df['core_strength_v2'] == strength_input.value].reset_index(drop=True)
    current_idx = 0
    display_image()

def on_prev_click(b):
    global current_idx
    if current_idx > 0:
        current_idx -= 1
        display_image()

def on_next_click(b):
    global current_idx
    if current_idx < len(filtered_df) - 1:
        current_idx += 1
        display_image()

def on_search(change):
    global current_idx
    search_query = search_input.value.strip()
    search_stem = Path(search_query).stem

    # 필터링된 데이터 내에서 검색
    match = filtered_df[filtered_df['image_id'] == search_stem]
    if not match.empty:
        current_idx = match.index[0]
        display_image()
    else:
        with out:
            print(f"'{search_query}'를 현재 필터 내에서 찾을 수 없습니다. (ID 확인 필요)")

# 이벤트 연결
strength_input.observe(on_filter_change, names='value')
prev_btn.on_click(on_prev_click)
next_btn.on_click(on_next_click)
search_input.on_submit(on_search)

# 초기 로드
filtered_df = df[df['core_strength_v2'] == strength_input.value].reset_index(drop=True)

# 레이아웃 배치 및 출력
controls = widgets.HBox([strength_input, prev_btn, next_btn])
display(controls, search_input, out)
display_image()

Text(value='', description='검색:', placeholder='파일명.png 입력 후 엔터', style=DescriptionStyle(description_width='ini…

Output()